# CMT Production Training
**Bayesian-optimized parameters** — ready to run on Colab T4/A100.

Run cells top to bottom. Checkpoints are saved directly to Google Drive every epoch.

In [ ]:
# 1. Setup
!git clone https://github.com/C0d3Crush/xray-vessel-inpainting.git
%cd xray-vessel-inpainting
!pip install -q torch torchvision matplotlib pillow opencv-python scikit-image pycocotools pandas numpy scipy
print('Setup complete')

In [ ]:
# 2. Mount Google Drive and extract ARCADE dataset
from google.colab import drive
import os
drive.mount('/content/drive')

ARCADE_ZIP = '/content/drive/MyDrive/arcade.zip'
!unzip -q -o "$ARCADE_ZIP" -d /content/xray-vessel-inpainting/data/
!find data/arcade -name '*.json' | head -5
print('ARCADE dataset extracted')


In [ ]:
# 3. Auto-detect ARCADE dataset paths
import os, glob, json
from collections import defaultdict

STENOSIS_CAT = 26  # same filter as load_coco_annotations in utils.py

def count_usable_annotations(ann_path):
    """Count images with at least one non-stenosis annotation."""
    with open(ann_path) as f:
        coco = json.load(f)
    by_img = defaultdict(list)
    for a in coco.get('annotations', []):
        if a['category_id'] != STENOSIS_CAT:
            by_img[a['image_id']].append(a)
    return sum(1 for img in coco.get('images', []) if by_img[img['id']])

def find_arcade_paths():
    candidates = []
    for root in ['/content/xray-vessel-inpainting', '/content', '.']:
        for match in glob.glob(f'{root}/**/train.json', recursive=True):
            if any(k in match for k in ('arcade', 'syntax', 'stenosis')):
                ann_dir = os.path.dirname(match)
                img_dir = os.path.join(os.path.dirname(ann_dir), 'images')
                if os.path.isdir(img_dir):
                    usable = count_usable_annotations(match)
                    candidates.append((match, img_dir, usable))
    # sort by usable annotations descending — syntax (vessel polygons) will rank first
    candidates.sort(key=lambda c: -c[2])
    return candidates

candidates = find_arcade_paths()
if not candidates:
    raise RuntimeError('ARCADE dataset not found — make sure Cell 2 ran successfully.')

print('Candidate annotation files (sorted by usable annotations):')
for ann, img, n in candidates:
    print(f'  {n:4d} usable images  {ann}')

# Pick the best candidate
best_ann, best_img, best_n = candidates[0]
if best_n == 0:
    raise RuntimeError(
        'No usable annotations found — all annotations are category 26 (stenosis).\n'
        'Make sure arcade.zip contains the SYNTAX dataset (vessel polygons), not only stenosis.'
    )

TRAIN_ANN = best_ann
TRAIN_IMG = best_img
VAL_ANN   = best_ann.replace('train/annotations/train.json', 'val/annotations/val.json')
VAL_IMG   = best_img.replace('train/images', 'val/images')

print(f'\nSelected:')
print(f'  TRAIN_IMG: {TRAIN_IMG}')
print(f'  TRAIN_ANN: {TRAIN_ANN}  ({best_n} usable images)')


In [ ]:
# 4. Imports, device, and annotation sanity-check
import sys, os, csv, json, subprocess
from collections import defaultdict

import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name} ({props.total_memory / 1e9:.1f} GB)')

STENOSIS_CAT = 26

def check_annotations(ann_path, label='dataset'):
    """Count usable (non-stenosis) annotated images — mirrors load_coco_annotations filter."""
    pkl_path = ann_path.replace('.json', '.pkl')
    if os.path.exists(pkl_path):
        import pickle
        with open(pkl_path, 'rb') as f:
            cached = pickle.load(f)
        n = len(cached.get('image_ids', []))
        print(f'  {label}: {n} usable images (from .pkl cache)')
        if n == 0:
            print(f'  WARNING: stale cache — delete it: !rm {pkl_path}')
        return n
    with open(ann_path) as f:
        coco = json.load(f)
    by_img = defaultdict(list)
    for a in coco.get('annotations', []):
        if a['category_id'] != STENOSIS_CAT:   # same filter as utils.py
            by_img[a['image_id']].append(a)
    n = sum(1 for img in coco.get('images', []) if by_img[img['id']])
    total = len(coco.get('images', []))
    all_anns = len(coco.get('annotations', []))
    filtered = sum(1 for a in coco.get('annotations', []) if a['category_id'] == STENOSIS_CAT)
    print(f'  {label}: {n}/{total} images have usable annotations  '
          f'(total anns: {all_anns}, stenosis filtered: {filtered})')
    return n

print('\nAnnotation check (stenosis cat-26 excluded, same as training):')
train_n = check_annotations(TRAIN_ANN, 'train')
val_n   = check_annotations(VAL_ANN,   'val')
if train_n == 0 or val_n == 0:
    raise RuntimeError(
        'Dataset has 0 usable images after filtering stenosis annotations.\n'
        'Check Cell 3 output — your arcade.zip may only contain the stenosis sub-challenge.\n'
        'You need the SYNTAX dataset (vessel polygon annotations).'
    )
print('\nDataset OK — ready to train.')


In [ ]:
# 5. Production training with Bayesian-optimized parameters
import shutil, subprocess, sys, csv, os
from datetime import datetime

# --- Optimized hyperparameters ---
BEST_PARAMS = {
    'lr':                1e-4,
    'batch_size':        6,
    'patches_per_image': 11,
    'foreground_prob':   0.7737056704131198,
    'max_shapes':        6,
    'perceptual_weight': 0.25,   # VGG16 perceptual loss for detail (0 = disabled)
}

EPOCHS       = 50
INPUT_SIZE   = 64
LOCAL_OUTPUT = 'checkpoints'

# Each run gets its own timestamped folder on Drive — previous runs are never overwritten
RUN_ID       = datetime.now().strftime('%Y%m%d_%H%M')
DRIVE_OUTPUT = f'/content/drive/MyDrive/CMT/runs/{RUN_ID}'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Fresh log: old logs have a different column layout (no hole metrics)
stale_log = os.path.join(LOCAL_OUTPUT, 'training_log.csv')
if os.path.exists(stale_log):
    os.remove(stale_log)

print(f'Run ID:  {RUN_ID}')
print(f'Drive:   {DRIVE_OUTPUT}')
print(f'epochs={EPOCHS}, lr={BEST_PARAMS["lr"]:.0e}, bs={BEST_PARAMS["batch_size"]}, '
      f'ppi={BEST_PARAMS["patches_per_image"]}, fg={BEST_PARAMS["foreground_prob"]:.4f}, '
      f'shapes={BEST_PARAMS["max_shapes"]}')
print()

cmd = [
    sys.executable, 'src/train.py',
    '--train_img',         TRAIN_IMG,
    '--train_ann',         TRAIN_ANN,
    '--val_img',           VAL_IMG,
    '--val_ann',           VAL_ANN,
    '--epochs',            str(EPOCHS),
    '--batch_size',        str(BEST_PARAMS['batch_size']),
    '--lr',                str(BEST_PARAMS['lr']),
    '--input_size',        str(INPUT_SIZE),
    '--device',            device,
    '--output_dir',        LOCAL_OUTPUT,
    '--patches_per_image', str(BEST_PARAMS['patches_per_image']),
    '--foreground_prob',   str(BEST_PARAMS['foreground_prob']),
    '--max_shapes',        str(BEST_PARAMS['max_shapes']),
    '--perceptual_weight', str(BEST_PARAMS['perceptual_weight']),
    '--save_every',        '1',    # save epoch checkpoint every epoch
    '--keep_checkpoints',  '5',    # local: keep last 5 (saves Colab disk)
    '--drive_dir',         DRIVE_OUTPUT,  # Drive: keeps ALL epochs
]
if device == 'cuda':
    cmd.append('--amp')

# Stream output live AND capture for error reporting
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
captured = []
for line in proc.stdout:
    print(line, end='', flush=True)
    captured.append(line)
proc.wait()

if proc.returncode != 0:
    print(f'\nTraining failed (exit {proc.returncode})')
    print('--- last 50 lines ---')
    print(''.join(captured[-50:]))
else:
    log_path = os.path.join(LOCAL_OUTPUT, 'training_log.csv')
    if os.path.exists(log_path):
        with open(log_path) as f:
            rows = list(csv.DictReader(f))
        if rows:
            last = rows[-1]
            print(f'\nTraining complete!')
            print(f'  Final PSNR:      {float(last.get("val_psnr", 0)):.2f} dB (full image)')
            print(f'  Final hole PSNR: {float(last.get("val_hole_psnr", 0)):.2f} dB (inpainted region only)')
            print(f'  Final SSIM:      {float(last.get("val_ssim", 0)):.4f} (full image)')
            print(f'  Final hole SSIM: {float(last.get("val_hole_ssim", 0)):.4f} (inpainted region only)')
            print(f'  Final RMSE: {last.get("val_rmse", "n/a")}')
    print(f'\nCheckpoints on Drive: {DRIVE_OUTPUT}')
    print(f'  epoch_001.pth ... epoch_{EPOCHS:03d}.pth  (all epochs)')
    print(f'  best.pth                                   (best hole PSNR)')
    print(f'  training_log.csv                           (all metrics)')


In [ ]:
# 6. Plot training curves
log_path = os.path.join(LOCAL_OUTPUT, 'training_log.csv')
if os.path.exists(log_path):
    !python scripts/plot_training.py {log_path}
    from IPython.display import Image, display
    plot_files = glob.glob('checkpoints/training_curves*.png')
    if plot_files:
        display(Image(sorted(plot_files)[-1]))
else:
    print('No training log found yet.')